# 파이프라인 전체 기능 테스트

청킹(`src/pipeline/chuncking.py`) → 임베딩/점수 전처리(`src/pipeline/embeddings.py` +
`src/pipeline/gisting.py`)의
현재 구현된 모든 기능을 확인한다.

`paginate_semantic` 를 활용한 페이지네이션

| 섹션 | 내용 | API 필요 |
|---|---|---|
| §1 | 청킹 — plain text 어댑터, `paginate_semantic` 페이지네이션, 원문 위치 역추적, 대사 보존 확인 | ❌ (mock) / ✅ (§1 끝 실제 API 데모) |
| §2 | 임베딩/점수 전처리 — mock 임베딩으로 청크 내부 중요도·유사도 흐름 확인 | ❌ |
| §3 | 실제 NIM 임베딩 API — 임베딩 호출, 유사도 점수 분석 | ✅ `NVIDIA_NIM_API_KEY` |

In [1]:
import os
import sys
from pathlib import Path

# src/ 디렉토리가 보일 때까지 위로 올라가 프로젝트 루트를 sys.path에 추가
# (import가 가능해지면 나머지 세팅은 src.notebook_setup.setup_project()에 위임 —
# 작업 디렉토리 고정, .env 로드, API 키 인식 여부 출력을 한 번에 처리)
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()
API_KEY_SET = bool(os.environ.get("NVIDIA_NIM_API_KEY"))

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env 로드: 성공 ✅
NVIDIA_NIM_API_KEY: 설정됨 ✅


## 함수 목록

이 노트북에서 다루는 모든 공개 함수 — 어느 섹션에서 데모하는지 함께 표시한다.

| 모듈 | 함수 | 역할 | 데모 위치 |
|---|---|---|---|
| `chuncking.py` | `plain_text_to_paragraphs` | plain text → `Paragraph` 리스트 | §1 plain text 어댑터 |
| `chuncking.py` | `paginate_semantic` | 이음매 유사도가 가장 낮은 지점에서 청크 분할 | §1 페이지네이션(mock/실제 API), §1 임베딩 실패 시 폴백 |
| `chuncking.py` | `_paginate_by_word_count_only`(내부 fallback) | 임베딩 실패 시 min/max words만으로 순서대로 분할 | §1 임베딩 실패 시 폴백 |
| `chuncking.py` | `chunk_containing_position` | 원문 char 위치 → 청크 역추적 | §1 원문 위치 역추적 |
| `embeddings.py` | `embed_texts` | NIM 임베딩 API 원시 호출 1회 | §3 실제 NIM 임베딩 API |
| `embeddings.py` | `embed_with_retry` | 재시도 포함 임베딩 호출 | §2 저수준 임베딩 유틸 직접 호출 |
| `embeddings.py` | `cosine_similarity` | 코사인 유사도 | §2 저수준 임베딩 유틸 직접 호출 |
| `embeddings.py` | `load_config` | YAML config 로드 | §1 전반(청크/임베딩 설정 로드) |
| `embeddings.py` | `EmbeddingAPIError` | 임베딩 API 실패 예외 | §1 임베딩 실패 시 폴백 |
| `gisting.py` | `embed_chunks` | 청크 리스트를 배치+재시도로 임베딩 | §2 청크 배치 임베딩 |
| `gisting.py` | `split_into_sentences` | 텍스트 → 문장 리스트 | §2 문장 분리 |
| `gisting.py` | `score_chunk_sentences` | 청크 "내부" 문장별 중요도 점수 | §2 청크 내부 중요도 지표 |

## 1. 청킹 (`chuncking.py`)

자르는 기준(우선순위): ① 문장(kss) 내부는 절대 안 자름 → ② 인용부호가 안 닫힌 대사 묶음은
한 덩어리로 유지 → ③ 대사 묶음이 max_words를 넘으면 문장 단위로 되돌림 → ④ `<p>` 경계는 후보
지점일 뿐 → ⑤ scene_break 문단만 하드 경계 → ⑥ min_words~max_words 구간의 이음매 후보 중
임베딩 유사도가 가장 낮은 지점을 청크 경계로 선택(`paginate_semantic`). 상세 근거는
`docs/pipelines.md` §청킹 참고.

### plain text 어댑터 (`plain_text_to_paragraphs`)

sample01은 빈 줄(`\n\s*\n+`)로 문단이 구분되는 plain text — 실제 운영 입력 형식. 블록 내부의
단일 개행은 문단 경계가 아니라 줄바꿈이라 공백으로 정규화된다. 장면 구분 기호로만 된 문단은
`is_scene_break`로 표시되어 페이지네이션에서 하드 경계가 된다.

In [3]:
import yaml

from src.pipeline.chuncking import (
    chunk_containing_position,
    paginate_semantic,
    plain_text_to_paragraphs,
)
from src.pipeline.embeddings import load_config

with open("data/sample/sample01.txt", encoding="utf-8") as f:
    raw = f.read()

paragraphs = plain_text_to_paragraphs(raw)
print("문단 수:", len(paragraphs), "\n")
for p in paragraphs:
    flag = " [scene_break]" if p.is_scene_break else ""
    print(f"[{p.index:2d}] offset={p.char_offset:5d} words={len(p.text.split()):4d}{flag} | {p.text[:40]}")

문단 수: 176 

[ 0] offset=    0 words=  80 | <도넛에 관한 농담> 해성 아파트의 주차장은 윤재가 묵고 있는 101동 
[ 1] offset=  326 words=  51 | 주차장 끝에 있는 놀이터를 보기 위해 싱크대 위로 몸을 뻗었다. 부엌 창
[ 2] offset=  520 words=  42 | 시간은 이미 새벽 두시를 넘기고 있었고 노인은 나올 기미가 없었다. 패브
[ 3] offset=  697 words=   1 [scene_break] | *
[ 4] offset=  703 words=  26 | 콜라 작가는 따뜻한 아메리카노 세 잔과 함께 이번에도 도넛을 추가로 주문
[ 5] offset=  801 words=   6 | “계산은 나중에 오시는 분이 해주실 거예요.”
[ 6] offset=  828 words=  58 | 한두 번이 아니었으니 직원도 군말 없이 돌아서서 커피를 내리기 시작했다.
[ 7] offset= 1054 words=   5 | “저 사람 좀 재밌지 않아요?”
[ 8] offset= 1073 words=  49 | 뒤따라 앉은 콜라 작가가 창밖을 가리켰다. 반바지 차림의 한 남자가 슬리
[ 9] offset= 1262 words=  16 | 별장도 아니고 세컨드 하우스라나? 머리에 피도 안 말라 보이는 애들이 돈
[10] offset= 1325 words=  23 | 최근 들어 원도심의 빈집 매매를 문의하는 외지인들이 많아졌다며 못마땅한 
[11] offset= 1425 words=  55 | 윤재가 청령시에 내려와 처음으로 원도심을 둘러봤던 때는 세 달 전인 11
[12] offset= 1640 words=   6 | 하필 주말이라 이런 거지, 평일엔 휑해요.
[13] offset= 1665 words=  24 | 윤재와 콜라 작가에게 길을 안내하던 장 주무관은 오해하지 말라는 투로 말
[14] offset= 1766 words=  30 | 지난여름 윤재는 청령시 도시재생과의 문자를 받고 나서

### 페이지네이션 (`paginate_semantic`) — mock 임베딩

`configs/chunking.yaml`의 min/max_words가 이음매 탐색 구간의 하한/상한이 된다. mock 임베딩으로
흐름을 먼저 확인하고, 실제 API 데모는 이 섹션 끝에 있다.

In [4]:
chunk_cfg = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))
gist_cfg = load_config("configs/importance_filter.yaml")

mock_cfg = dict(gist_cfg)
mock_cfg["api_key_env"] = "MOCK_EMBED_KEY"
os.environ["MOCK_EMBED_KEY"] = "offline-dummy"


def fake_embed(texts, input_type, model, truncate, api_key, timeout=30.0, dimensions=None):
    """문단 텍스트 해시 기반 가짜 벡터 — 실제 의미는 없지만 문단마다 값이 달라 이음매 유사도가 갈린다."""
    return [[hash(t) % 97 / 97.0, hash(t[::-1]) % 89 / 89.0, 0.1] for t in texts]


chunks = paginate_semantic(
    paragraphs,
    min_words=chunk_cfg["min_words"],
    max_words=chunk_cfg["max_words"],
    granularity="paragraph",
    config=mock_cfg,
    embed_fn=fake_embed,
)

print(f"청크 수: {len(chunks)}  (min_words={chunk_cfg['min_words']}, max_words={chunk_cfg['max_words']})\n")
for c in chunks:
    print(
        f"[{c.index:2d}] words={len(c.text.split()):3d} "
        f"chars=({c.char_start:5d}~{c.char_end:5d}) 문단={c.paragraph_indices} | {c.text[:35]}"
    )

청크 수: 38  (min_words=150, max_words=350)

[ 0] words=174 chars=(    0~  698) 문단=[0, 1, 2, 3] | <도넛에 관한 농담> 해성 아파트의 주차장은 윤재가 묵고 있는 
[ 1] words=244 chars=(  703~ 1663) 문단=[4, 5, 6, 7, 8, 9, 10, 11, 12] | 콜라 작가는 따뜻한 아메리카노 세 잔과 함께 이번에도 도넛을 추
[ 2] words=245 chars=( 1665~ 2743) 문단=[13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26] | 윤재와 콜라 작가에게 길을 안내하던 장 주무관은 오해하지 말라는
[ 3] words=220 chars=( 2745~ 3654) 문단=[27, 28, 29, 30, 31, 32, 33, 34, 35, 36] | “아는 PD님 통해서 이미 다음달 방송 날짜 다 잡아놨어요. 그
[ 4] words=322 chars=( 3659~ 4964) 문단=[37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47] | 의자는 항상 새벽을 틈타 사라졌다. 사라진 의자 중엔 윤재가 서
[ 5] words=208 chars=( 4966~ 5789) 문단=[48, 49, 50, 51, 52, 53] | 콜라 작가는 아직 손대지 않은 윤재 몫의 커피와 도넛을 건넸다.
[ 6] words=254 chars=( 5791~ 6822) 문단=[54, 55, 56, 57, 58] | 투박하고 정직한 1호와 2호. 기하학적인 조형이 돋보이는 3호와
[ 7] words=147 chars=( 6824~ 7436) 문단=[59, 60, 61, 62, 63] | 세 면이 담벼락으로 둘러싸인 쌈지 공간이 나왔다. 그늘진 한쪽 
[ 8] words=199 chars=( 7441~ 8259) 문단=[64, 65, 66, 67, 68, 69, 70] | 원도심 바깥 능선 너머로 어스름이 깔렸다. 가까이

### 대사가 청크 경계에서 안 잘리는지 확인

`paginate_semantic` 내부의 `_dialogue_groups`가 인용부호(`“” ‘’ "" ''`)로 묶인 문장들을 하나의
청크 경계 안에 유지하는지, 실제 청크 전체를 훑어 검증한다. 아포스트로피(`I'm`, `don't`)는
인용부호로 오인하지 않도록 제외하고 센다.

In [5]:
import re


def _quotes_balanced(text: str) -> bool:
    stripped = re.sub(r"(?<=[A-Za-z0-9])[’\']", "", text)  # 아포스트로피 제외
    return stripped.count("‘") == stripped.count("’") and stripped.count("“") == stripped.count("”")


unbalanced = [c.index for c in chunks if not _quotes_balanced(c.text)]
print("인용부호가 안 닫힌 채 끝나는 청크:", unbalanced if unbalanced else "없음 ✅")

인용부호가 안 닫힌 채 끝나는 청크: 없음 ✅


### 원문 위치 → 청크 역추적 (`chunk_containing_position`)

정답 근거가 실제로 어느 청크에 있었는지 사후 분석할 때 쓴다.

In [7]:
needle = "조경수들과 띄엄띄엄 주차된 차들" #검색에 사용할 텍스트 내 문자열 일부
pos = raw.find(needle)
hit = chunk_containing_position(chunks, pos)
print(f"원문 {pos}번째 문자({needle!r} 등장 위치) → 청크 {hit.index}")
print("청크 앞부분:", hit.text[:80])

# 원문 조회가 실제로 맞는지 눈으로 확인
print("원문 해당 구간:", raw[hit.char_start : hit.char_start + 80].replace("\n", " "))

# 범위 밖 위치는 None
print("범위 밖 조회 결과:", chunk_containing_position(chunks, 10**9))

원문 56번째 문자('조경수들과 띄엄띄엄 주차된 차들' 등장 위치) → 청크 0
청크 앞부분: <도넛에 관한 농담> 해성 아파트의 주차장은 윤재가 묵고 있는 101동 4층에서도 한눈에 들어왔다. 조경수들과 띄엄띄엄 주차된 차들 사이로 10
원문 해당 구간: <도넛에 관한 농담> 해성 아파트의 주차장은 윤재가 묵고 있는 101동 4층에서도 한눈에 들어왔다. 조경수들과 띄엄띄엄 주차된 차들 사이로 10
범위 밖 조회 결과: None


### 페이지네이션 — 실제 API (`paginate_semantic`)

쿼터 절약을 위해 앞쪽 문단 일부만 사용

In [8]:
if not API_KEY_SET:
    print("NVIDIA_NIM_API_KEY가 없어 건너뜁니다.")
else:
    sample_paragraphs = paragraphs[:8]  # 쿼터 절약
    chunks_real = paginate_semantic(
        sample_paragraphs,
        min_words=chunk_cfg["min_words"],
        max_words=chunk_cfg["max_words"],
        granularity="paragraph",
        config=gist_cfg,
    )
    print(f"입력 문단 {len(sample_paragraphs)}개 → 청크 {len(chunks_real)}개\n")
    for c in chunks_real:
        print(f"[{c.index}] words={len(c.text.split())} 문단={c.paragraph_indices} | {c.text[:50]}")

입력 문단 8개 → 청크 2개

[0] words=175 문단=[0, 1, 2, 3] | <도넛에 관한 농담> 해성 아파트의 주차장은 윤재가 묵고 있는 101동 4층에서도 한눈에 
[1] words=95 문단=[4, 5, 6, 7] | 콜라 작가는 따뜻한 아메리카노 세 잔과 함께 이번에도 도넛을 추가로 주문했다. 혼자만 먹기


### 임베딩 실패 시 폴백 (`_paginate_by_word_count_only`)

`paginate_semantic`이 임베딩 API를 계속 실패하면 `config["on_error"]`에 따라 동작이 갈린다 —
`pass_through`면 유사도 판단 없이 `_paginate_by_word_count_only`(min/max words만으로 순서대로
채우다 넘치면 끊는 내부 fallback, 구 `paginate_fixed`)로 완전히 대체하고, `raise`면
`EmbeddingAPIError`를 그대로 전파한다.

In [9]:
from src.pipeline.chuncking import _paginate_by_word_count_only
from src.pipeline.embeddings import EmbeddingAPIError


def always_fails(*args, **kwargs):
    raise EmbeddingAPIError("simulated failure")


fail_cfg = dict(mock_cfg)
fail_cfg["max_retries"] = 0
fail_cfg["on_error"] = "pass_through"

fallback_chunks = paginate_semantic(
    paragraphs,
    min_words=chunk_cfg["min_words"],
    max_words=chunk_cfg["max_words"],
    granularity="paragraph",
    config=fail_cfg,
    embed_fn=always_fails,
)
direct_chunks = _paginate_by_word_count_only(paragraphs, chunk_cfg["min_words"], chunk_cfg["max_words"])

print("pass_through 폴백 청크 수:", len(fallback_chunks), "vs _paginate_by_word_count_only 직접 호출:", len(direct_chunks))
print("텍스트까지 완전히 동일:", [c.text for c in fallback_chunks] == [c.text for c in direct_chunks])

# on_error="raise"면 EmbeddingAPIError가 실제로 발생하는지 확인
fail_cfg_raise = dict(fail_cfg)
fail_cfg_raise["on_error"] = "raise"
try:
    paginate_semantic(
        paragraphs,
        min_words=chunk_cfg["min_words"],
        max_words=chunk_cfg["max_words"],
        granularity="paragraph",
        config=fail_cfg_raise,
        embed_fn=always_fails,
    )
    print("\nEmbeddingAPIError가 발생하지 않음 — 문제!")
except EmbeddingAPIError as e:
    print("\n의도대로 EmbeddingAPIError 발생:", e)

[embeddings] 임베딩 API 호출 실패 (재시도 0회 소진, 텍스트 16개, input_type='passage'): EmbeddingAPIError: simulated failure


pass_through 폴백 청크 수: 28 vs _paginate_by_word_count_only 직접 호출: 28
텍스트까지 완전히 동일: True

의도대로 EmbeddingAPIError 발생: 임베딩 API 호출이 재시도 후에도 실패해 의미 기반 페이지네이션을 진행할 수 없습니다. 원인: EmbeddingAPIError: simulated failure


[embeddings] 임베딩 API 호출 실패 (재시도 0회 소진, 텍스트 16개, input_type='passage'): EmbeddingAPIError: simulated failure


## 2. 임베딩/점수 전처리 (`embeddings.py` + `gisting.py`) — 오프라인, mock 임베딩

- `embed_with_retry` / `cosine_similarity` — 저수준 임베딩 유틸(embeddings.py), 직접 호출 데모.
- `embed_chunks` — 청크 리스트 배치 임베딩(gisting.py).
- `split_into_sentences` — 텍스트를 문장 리스트로 분리(gisting.py).
- `score_chunk_sentences` — 청크 "내부" 문장별 중요도 지표(청크 자기 자신의 임베딩과 문장
  임베딩의 코사인 유사도). 아무것도 버리지 않고 점수만 반환.

청크-질문 유사도(`score_chunks`, `embed_query`, `ScoredChunk`)는 이번 스코프에서 제외됐다 —
gisting.py에서 삭제됐고, mock/실제 API 데모 모두 없다.

실제 압축(청크 요약)은 되뇌기를 구현할 때 별도 모듈로 추가할 예정.

### 저수준 임베딩 유틸 직접 호출 (`embed_with_retry`, `cosine_similarity`)

`embed_with_retry`로 텍스트를 직접 임베딩하고, 그 벡터끼리 `cosine_similarity`를 계산한다.
화제가 비슷한 문장끼리는 유사도가 높고, 다른 문장과는 낮게 나오는지 확인한다.

In [10]:
from src.pipeline.embeddings import cosine_similarity, embed_with_retry


def fake_embed_by_keyword(texts, input_type, model, truncate, api_key, timeout=30.0, dimensions=None):
    """'고양이' 포함 여부로 벡터를 갈라 유사도 차이를 눈으로 보여주는 가짜 임베딩."""
    return [[1.0 if "고양이" in t else 0.0, 1.0 if "고양이" not in t else 0.0, 0.1] for t in texts]


sample_texts = ["고양이가 창밖을 바라본다.", "고양이는 낮잠을 잔다.", "주가 지수가 급락했다."]
vectors = embed_with_retry(sample_texts, "passage", mock_cfg, fake_embed_by_keyword)

print("반환된 벡터 수:", len(vectors))
for t, v in zip(sample_texts, vectors):
    print(f"  {v} | {t}")

sim_similar = cosine_similarity(vectors[0], vectors[1])  # 둘 다 '고양이' 관련
sim_different = cosine_similarity(vectors[0], vectors[2])  # 화제가 다름

print(f"\n'고양이' 문장끼리 유사도: {sim_similar:.4f} (높음)")
print(f"화제가 다른 문장과의 유사도: {sim_different:.4f} (낮음)")

반환된 벡터 수: 3
  [1.0, 0.0, 0.1] | 고양이가 창밖을 바라본다.
  [1.0, 0.0, 0.1] | 고양이는 낮잠을 잔다.
  [0.0, 1.0, 0.1] | 주가 지수가 급락했다.

'고양이' 문장끼리 유사도: 1.0000 (높음)
화제가 다른 문장과의 유사도: 0.0099 (낮음)


### 청크 배치 임베딩 (`embed_chunks`)

청크 리스트 전체를 한 번에 `embed_chunks`에 넣어 배치+재시도가 어떻게 동작하는지 보여준다.
그 결과 중 하나를 아래 `score_chunk_sentences` 데모의 `chunk_embedding` 인자로 그대로 이어서
사용한다.

In [11]:
from src.pipeline.gisting import embed_chunks

chunk_embeddings = embed_chunks(chunks, mock_cfg, embed_fn=fake_embed)
print(f"청크 {len(chunks)}개 → 임베딩 {len(chunk_embeddings)}개 (batch_size={mock_cfg['batch_size']})")
print("첫 번째 청크 임베딩:", chunk_embeddings[0])

청크 38개 → 임베딩 38개 (batch_size=16)
첫 번째 청크 임베딩: [0.0, 0.20224719101123595, 0.1]


### 문장 분리 (`split_into_sentences`)

청크 하나의 `text`를 `split_into_sentences`에 직접 넣어 문장 리스트를 확인한다.

In [12]:
from src.pipeline.gisting import split_into_sentences

sample_sentences = split_into_sentences(chunks[0].text)
print(f"청크 0을 문장 {len(sample_sentences)}개로 분리:\n")
for s in sample_sentences[:5]:
    print(" -", s[:60])

청크 0을 문장 17개로 분리:

 - <도넛에 관한 농담> 해성 아파트의 주차장은 윤재가 묵고 있는 101동 4층에서도 한눈에 들어왔다.
 - 조경수들과 띄엄띄엄 주차된 차들 사이로 102동 노인이 만든 구조물 네 개가 간신히 윤곽을 드러냈다.
 - 큐브, 망루, 캣 타워, 천칭. 윤재는 콜라 작가가 붙인 이 직관적인 이름들이 썩 마음에 들지 않았지만 지금
 - 둔탁한 정육면체 프레임을 가진 1호. 높게 치솟은 2호. 정글짐처럼 복잡하게 얽혀 있는 3호와 고고하게 양팔
 - 주차장 끝에 있는 놀이터를 보기 위해 싱크대 위로 몸을 뻗었다.


### 청크 내부 중요도 지표 (`score_chunk_sentences`)

한 청크 안에서 어떤 문장이 "중요한지"를 점수로만 보여준다 — 압축은 하지 않음.

In [13]:
from src.pipeline.gisting import score_chunk_sentences

sample_chunk = chunks[0]
chunk_embedding = chunk_embeddings[sample_chunk.index]  # 위 embed_chunks 결과를 그대로 재사용
scored = score_chunk_sentences(sample_chunk, chunk_embedding, config=mock_cfg, embed_fn=fake_embed)

print(f"청크 {sample_chunk.index} 문장별 중요도 (상위 5개):\n")
for sentence, score in sorted(scored, key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {score:.3f} | {sentence[:60]}")

청크 0 문장별 중요도 (상위 5개):

  0.936 | 시간은 이미 새벽 두시를 넘기고 있었고 노인은 나올 기미가 없었다.
  0.860 | 어제 그가 구해온 베이지색 패브릭 소파는 미끄럼틀 옆에 그대로 놓여 있었다.
  0.835 | 노인 역시 윤재만큼이나 최선을 다해 의자를 선별하고 있다
  0.801 | 허탈했지만 한편으론 다행이라고 여겼다.
  0.797 | 패브릭 소파는 선택받지 못했다.


## 3. 실제 NIM 임베딩 API

`NVIDIA_NIM_API_KEY`가 설정되어 있을 때만 실행된다. 쿼터 절약을 위해 앞쪽 청크 일부만 쓴다.

In [14]:
if not API_KEY_SET:
    print("NVIDIA_NIM_API_KEY가 없어 건너뜁니다.")
else:
    from src.pipeline.embeddings import embed_texts

    emb = embed_texts(
        ["테스트 문장입니다."],
        input_type="passage",
        model=gist_cfg["model"],
        truncate=gist_cfg["truncate"],
        api_key=os.environ["NVIDIA_NIM_API_KEY"],
    )
    print("모델:", gist_cfg["model"])
    print("임베딩 차원:", len(emb[0]), "| 앞 5개 값:", [round(v, 4) for v in emb[0][:5]])

모델: nvidia/llama-nemotron-embed-1b-v2
임베딩 차원: 2048 | 앞 5개 값: [0.0043, -0.0034, 0.0291, 0.0073, 0.0325]
